## Imports

In [1]:
import xarray as xr

In [2]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

## Open the data

In [3]:
# data in persistent bucket

target_url ='gs://leap-persistent/cspencerjones/hero-calc/masked_hist_renorm_3hr.zarr'
target_wet_area = 'gs://leap-persistent/cspencerjones/hero-calc/wet_area.zarr'

In [4]:
hist_ds = xr.open_dataset(target_url, engine="zarr", chunks={})
ds_wet = xr.open_dataset(target_wet_area, engine="zarr", chunks={})
hist_ds = hist_ds.assign_coords(region_wet_area=('region', ds_wet['wet_area'].data))
hist_ds = hist_ds.assign_coords(area_fraction=('region', ds_wet['area_fraction'].data))

In [5]:
hist_ds['histogram_vort_strain_div'].isel(time=0).nbytes / 1e9

1.678950504

In [6]:
from xarray.indexes import PandasIndex

# have to add the index ourselves manually for some reason
hist_ds = hist_ds.set_xindex('region_num', PandasIndex)

## Expectation values

## Latitude-dependent mean

We want to find the mean of all histograms, but defining the mean in such a way that it varies with latitude but not longitude, i.e. not across all regions.

In [9]:
# find the latitude of the centrepoint of each region polygon
lat = hist_ds['vertices_latitude'].mean(dim='vertices').rename('center_latitude')

In [15]:
hist_ds['vertices_latitude']

<xarray.DataArray 'vertices_latitude' (vertices: 4, region: 437)> Size: 7kB
dask.array<open_dataset-vertices_latitude, shape=(4, 437), dtype=float32, chunksize=(4, 437), chunktype=numpy.ndarray>
Coordinates:
    j_region_coarse     (region) int64 3kB 4 5 6 7 5 6 7 5 6 ... 3 4 6 7 0 1 6 7
    vertices_longitude  (vertices, region) float32 7kB dask.array<chunksize=(4, 437), meta=np.ndarray>
  * region_num          (region) int64 3kB 4 5 6 7 13 14 ... 791 792 793 798 799
    i_region_coarse     (region) int64 3kB 0 0 0 0 1 1 1 2 2 ... 2 2 2 2 3 3 3 3
    face                (region) int64 3kB 0 0 0 0 0 0 0 ... 12 12 12 12 12 12
    vertices_latitude   (vertices, region) float32 7kB dask.array<chunksize=(4, 437), meta=np.ndarray>
    region_wet_area     (region) float64 3kB 1.807e+11 2.148e+11 ... 1.78e+11
    area_fraction       (region) float64 3kB 0.9951 1.0 1.0 ... 0.8939 1.0
    lat                 (region) float32 2kB -73.97 -69.45 ... -73.68 -73.86
Dimensions without coordinates: vertices, region

In [10]:
hist_ds.coords['lat'] = lat.load()

In [11]:
lat_bin_edges = np.asarray([-80,-55,-30, -15, 15, 30, 55, 80])#np.arange(start=-80, stop=81, step=20)

In [21]:
#Rescale histograms to deal with missing points
hist_ds_normalized = (hist_ds['histogram_vort_strain_div'])/hist_ds['area_fraction']
hist_ds_normalized =(hist_ds_normalized.rename('probability_density')).to_dataset().merge(hist_ds['vertices_latitude'].rename('vert_latitude'))
hist_ds_normalized = hist_ds_normalized.drop_vars('vert_latitude')
hist_ds_normalized

<xarray.Dataset> Size: 62GB
Dimensions:              (region: 437, div_bin: 99, strain_bin: 49, time: 37,
                          vort_bin: 99, vertices: 4)
Coordinates: (12/13)
    j_region_coarse      (region) int64 3kB 4 5 6 7 5 6 7 5 ... 3 4 6 7 0 1 6 7
  * region_num           (region) int64 3kB 4 5 6 7 13 ... 791 792 793 798 799
    i_region_coarse      (region) int64 3kB 0 0 0 0 1 1 1 2 ... 2 2 2 2 3 3 3 3
    face                 (region) int64 3kB 0 0 0 0 0 0 0 ... 12 12 12 12 12 12
    region_wet_area      (region) float64 3kB 1.807e+11 2.148e+11 ... 1.78e+11
    area_fraction        (region) float64 3kB 0.9951 1.0 1.0 ... 0.8939 1.0
    ...                   ...
    vertices_longitude   (vertices, region) float32 7kB dask.array<chunksize=(4, 437), meta=np.ndarray>
    vertices_latitude    (vertices, region) float32 7kB dask.array<chunksize=(4, 437), meta=np.ndarray>
  * div_bin              (div_bin) float64 792B -4.949 -4.848 ... 4.848 4.949
  * strain_bin           (strain_bin) float64 392B 0.05102 0.1531 ... 4.949
  * time                 (time) datetime64[ns] 296B 2011-09-17T22:30:00 ... 2...
  * vort_bin             (vort_bin) float64 792B -4.949 -4.848 ... 4.848 4.949
Dimensions without coordinates: region, vertices
Data variables:
    probability_density  (time, region, vort_bin, strain_bin, div_bin) float64 62GB dask.array<chunksize=(1, 1, 99, 49, 99), meta=np.ndarray>

In [22]:
for t in range(0,37):
    hist_ds_save = hist_ds_normalized.isel(time=[t])
    for var in hist_ds_save .data_vars:
        if hist_ds_save [var].dtype == 'float64':
            hist_ds_save [var] = hist_ds_save[var].astype('float32')



    encoding = {
        var: dict(
            zlib=True,
            complevel=1,
            dtype='float32',
            chunksizes=(1, 23, 99, 49, 99)  # chunk size per dimension
        )
    for var in hist_ds_save.data_vars
    }
    hist_ds_save.to_netcdf('hist_save'+str(t).zfill(2) +'.nc', encoding=encoding)

In [23]:
check = xr.open_mfdataset('hist_save*.nc')

In [24]:
encoding = {
        var: dict(
            zlib=True,
            complevel=1,
            dtype='float32',
            chunksizes=(1, 23, 99, 49, 99)  # chunk size per dimension
        )
for var in check.data_vars
}
check.to_netcdf('hist_save.nc', encoding=encoding)